In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
import numpy as np

    

# SE LEE EL ARCHIVO RECIBIDO POR PARTE DE LAURA LIDER REGISTRO Y CONTROL

In [2]:
df_excel = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\ESTUDIANTES_V2\MALLAS\ARCHIVOS_MALLAS\MALLAS CUN ACTUALIZADO_AGOSTO_2026.xlsx', sheet_name='MALLAS CUN PRESENCIAL Y VIR ')
df_excel

c:\Users\alejandro_arboledas\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,TIPO DE REGISTRO,ESTADO,ESCUELAS,PROGRAMA PROFESIONAL,SEDE,MODALIDAD,PENSUM,COD PROGR TECNICO,RESOLUCION_TEC,RESUELVE_TEC,...,4 C,5 C,6 C,7 C,8 C,9 C,10 C,11,TOTAL CREDITOS MALLA,Unnamed: 47
0,RENOVACION,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,COMUNICACION SOCIAL,BARRANQUILLA,PRESENCIAL,3031,1TQ12,16932,2016-08-22 00:00:00,...,17.0,19.0,19.0,16.0,17.0,17.0,NaN,NaN,157.0,NaN
1,REGISTRO CALIFICADO,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3024,1TC11,3567,2009-06-01 00:00:00,...,20.0,18.0,19.0,18.0,20.0,18.0,20.0,NaN,190.0,NaN
2,RENOVACION,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3027,1TC11,3567,2009-06-01 00:00:00,...,20.0,18.0,21.0,21.0,18.0,19.0,18.0,NaN,192.0,NaN
3,RENOVACION,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3028,1TC11,10711,2016-06-01 00:00:00,...,20.0,20.0,20.0,19.0,17.0,18.0,NaN,NaN,169.0,NaN
4,REGISTRO CALIFICADO,VENCIDO,ESCUELA DE INGENIERIAS,INGENIERIA DE SISTEMAS,BOGOTA CENTRO,PRESENCIAL,2011,1TC03,421,2011-02-06 00:00:00,...,19.0,20.0,21.0,20.0,20.0,20.0,19.0,NaN,200.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,REGISTRO NUEVO,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,NEGOCIOS,NEIVA,PRESENCIAL,8015,1TV21,014015,2026-05-22 00:00:00,...,17.0,17.0,17.0,17.0,17.0,17.0,NaN,NaN,154.0,NaN
358,REGISTRO NUEVO,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,NEGOCIOS,SANTA MARTA,PRESENCIAL,8015,1TR28,014015,2026-05-22 00:00:00,...,17.0,17.0,17.0,17.0,17.0,17.0,NaN,NaN,154.0,NaN
359,REGISTRO NUEVO,VIGENTE,ESCUELA DE INGENIERIAS,INGENIERÍA QUÍMICA,BOGOTA CENTRO,HIBRIDO,2050,1TC77,017087,2026-06-30 00:00:00,...,17.0,18.0,17.0,18.0,18.0,18.0,NaN,NaN,156.0,NaN
360,REGISTRO NUEVO UNICO,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACIÓN DE EMPRESAS TURÍSTICAS Y HOTELERAS,BOGOTA CENTRO,PRESENCIAL,8037,1TC65,017078,2026-06-30 00:00:00,...,18.0,16.0,17.0,17.0,17.0,17.0,NaN,NaN,155.0,NaN


In [3]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel.columns:
    col_str = str(col)

    if col_str.startswith('TOTAL CREDITOS'):
        contador += 1
        if contador == 1:
            new_cols.append('TOTAL CREDITOS TEC')
        elif contador == 2:
            new_cols.append('TOTAL CREDITOS TNG')
        elif contador == 3:
            new_cols.append('TOTAL CREDITOS PRO')
        elif contador == 4:
            new_cols.append('TOTAL_CRED_PROF')   
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'TOTAL CREDITOS_{contador}')
    else:
        new_cols.append(col_str)

df_excel.columns = new_cols

In [4]:
# SE RENOMBRAN LAS COLUMNAS
df_excel = df_excel.rename(columns={'PROGRAMA PROFESIONAL':'PROGRAMA_PROFESIONAL','COD PROGR TECNICO' : 'COD_PROG_TECNICO', 'RESOLUCION_TEC':'RESOLUCION_TECNICO',
                                    'RESUELVE_TEC':'FECHA_INICIO_TEC','CODIGO SNIES TECNICO':'CODIGO_SNIES_TECNICO', 'FECHA SNIES VENCIMIENTO_TEC':'FECHA_FIN_TEC',
                                    'COD PROG TECNOLOGO':'COD_PROG_TECNOLOGO','RESUELVE_TNG':'FECHA_INICIO_TNG','CODIGO SNIES TECNOLOGICO':'CODIGO_SNIES_TECNOLOGICO',
                                    'FECHA SNIES VENCIMIENTO_TNG':'FECHA_FIN_TNG','COD PROG PROFESIONAL':'COD_PROG_PROFESIONAL','RESUELVE_PRO':'FECHA_INICIO_PRO',
                                    'CODIGO SNIES PROF':'CODIGO_SNIES_PROF','FECHA SNIES VENCIMIENTO_PRO':'FECHA_FIN_PRO','TECNICO NIVELES POR CICLO':'TECNICO_NIVELES',
                                    'TOTAL CREDITOS TEC':'TOTAL_CREDITOS_TECNICO','TECNOLOGO NIVELES POR CICLO':'TECNOLOGO_NIVELES','TOTAL CREDITOS TNG':'TOTAL_CREDITOS_TECNOLOGO',
                                    'PROFESIONAL NIVELES POR CICLO':'PROFESIONAL_NIVELES','TOTAL CREDITOS PRO':'TOTAL_CREDITOS_PROFESIONAL','11':'11C'
                                    })

In [5]:
df_excel.columns = df_excel.columns.str.replace(' ', '')

In [6]:
df_excel.columns

Index(['TIPODEREGISTRO', 'ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE',
       'MODALIDAD', 'PENSUM', 'COD_PROG_TECNICO', 'RESOLUCION_TECNICO',
       'FECHA_INICIO_TEC', 'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC',
       'NOMBRE_DEL_PROGRAMA_TEC', 'TITULO_OTORGADO_TEC', 'COD_PROG_TECNOLOGO',
       'RESOLUCION_TNG', 'FECHA_INICIO_TNG', 'CODIGO_SNIES_TECNOLOGICO',
       'FECHA_FIN_TNG', 'NOMBRE_DEL_PROGRAMA_TNG', 'TITULO_OTORGADO_TNG',
       'COD_PROG_PROFESIONAL', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', 'TECNOLOGO_NIVELES',
       'TOTAL_CREDITOS_TECNOLOGO', 'PROFESIONAL_NIVELES',
       'TOTAL_CREDITOS_PROFESIONAL', '1C', '2C', '3C', '4C', '5C', '6C', '7C',
       '8C', '9C', '10C', '11C', 'TOTAL_CRED_PROF', 'Unnamed:47'],
      dtype='object')

In [7]:
# definicion de columnas
df_mallas_laura = df_excel[['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'PENSUM', 'COD_PROG_TECNICO', 'RESOLUCION_TECNICO', 'FECHA_INICIO_TEC',
       'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC', 'NOMBRE_DEL_PROGRAMA_TEC',
       'TITULO_OTORGADO_TEC', 'COD_PROG_TECNOLOGO', 'RESOLUCION_TNG',
       'FECHA_INICIO_TNG', 'CODIGO_SNIES_TECNOLOGICO', 'FECHA_FIN_TNG',
       'NOMBRE_DEL_PROGRAMA_TNG', 'TITULO_OTORGADO_TNG',
       'COD_PROG_PROFESIONAL', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', 'TECNOLOGO_NIVELES',
       'TOTAL_CREDITOS_TECNOLOGO', 'PROFESIONAL_NIVELES',
       'TOTAL_CREDITOS_PROFESIONAL', 'TOTAL_CRED_PROF', '1C', '2C', '3C', '4C',
       '5C', '6C', '7C', '8C', '9C', '10C', '11C']].copy()

In [8]:
#transformacion de fechas
df_mallas_laura['FECHA_INICIO_TEC'] = pd.to_datetime(df_mallas_laura['FECHA_INICIO_TEC'], errors='coerce')
df_mallas_laura['FECHA_FIN_TEC'] = pd.to_datetime(df_mallas_laura['FECHA_FIN_TEC'], errors='coerce')
df_mallas_laura['FECHA_INICIO_TNG'] = pd.to_datetime(df_mallas_laura['FECHA_INICIO_TNG'], errors='coerce')
df_mallas_laura['FECHA_FIN_TNG'] = pd.to_datetime(df_mallas_laura['FECHA_FIN_TNG'], errors='coerce')
df_mallas_laura['FECHA_INICIO_PRO'] = pd.to_datetime(df_mallas_laura['FECHA_INICIO_PRO'], errors='coerce')
df_mallas_laura['FECHA_FIN_PRO'] = pd.to_datetime(df_mallas_laura['FECHA_FIN_PRO'], errors='coerce')


df_mallas_laura['FECHA_INICIO_TEC'] = df_mallas_laura['FECHA_INICIO_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura['FECHA_FIN_TEC'] = df_mallas_laura['FECHA_FIN_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura['FECHA_INICIO_TNG'] = df_mallas_laura['FECHA_INICIO_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura['FECHA_FIN_TNG'] = df_mallas_laura['FECHA_FIN_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura['FECHA_INICIO_PRO'] = df_mallas_laura['FECHA_INICIO_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura['FECHA_FIN_PRO'] = df_mallas_laura['FECHA_FIN_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')


In [9]:
df_excel_distancia = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\ESTUDIANTES_V2\MALLAS\ARCHIVOS_MALLAS\MALLAS CUN ACTUALIZADO_JUNIO_2026.xlsx', sheet_name='DISTANCIA')
df_excel_distancia

,ESTADO,ESCUELAS,PROGRAMA,SEDE,MODALIDAD,PENSUM,TECNICO,RESOLUCION,RESUELVE,CODIGO SNIES TECNICO,...,3 C,4 C,5 C,6 C,7 C,8 C,9 C,10 C,TOTAL CREDITO MALLA,Unnamed: 45
0,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,BOGOTA CENTRO,DISTANCIA,1014,TBC04,8193,2007-12-28,2380,...,17.0,18.0,16.0,18.0,18.0,18.0,18.0,20.0,174,NaN
1,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,BOGOTA CENTRO,DISTANCIA,1015,TBC04,NaN,NaT,2380,...,17.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,69,NaN
2,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,BOGOTA CENTRO,DISTANCIA,1018,TBC04,4215,2012-04-20,2380,...,15.0,18.0,19.0,16.0,16.0,19.0,16.0,16.0,175,NaN
3,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,BOGOTA CENTRO,DISTANCIA,1028,TBC04,4215,2012-04-20,2380,...,16.0,16.0,21.0,19.0,16.0,21.0,14.0,16.0,175,NaN
4,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,BOGOTA CENTRO,DISTANCIA,1035,TBD06,07754,2016-04-21,105498,...,19.0,19.0,20.0,17.0,14.0,16.0,17.0,NaN,158,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
920,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,BARRANQUILLA,DISTANCIA,1080,TBQ12,´021098,2025-11-01,118370,...,18.0,18.0,18.0,18.0,18.0,18.0,17.0,NaN,160,NaN
921,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,MONTERIA,DISTANCIA,1080,TMO14,´021098,2025-11-02,118370,...,18.0,18.0,18.0,18.0,18.0,18.0,17.0,NaN,160,NaN
922,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,SANTA MARTA,DISTANCIA,1080,TSM14,´021098,2025-11-03,118370,...,18.0,18.0,18.0,18.0,18.0,18.0,17.0,NaN,160,NaN
923,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMNISTRACION EMPRESAS,SINCELEJO,DISTANCIA,1080,TSJ12,´021098,2025-11-04,118370,...,18.0,18.0,18.0,18.0,18.0,18.0,17.0,NaN,160,NaN


In [10]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_distancia.columns:
    col_str = str(col)

    if col_str.startswith('RESOLUCION'):
        contador += 1
        if contador == 1:
            new_cols.append('RESOLUCION_TECNICO')
        elif contador == 2:
            new_cols.append('RESOLUCION_TNG')
        elif contador == 3:
            new_cols.append('RESOLUCION_PROF')
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'RESOLUCION_{contador}')
    else:
        new_cols.append(col_str)

df_excel_distancia.columns = new_cols

In [11]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_distancia.columns:
    col_str = str(col)

    if col_str.startswith('RESUELVE'):
        contador += 1
        if contador == 1:
            new_cols.append('FECHA_INICIO_TEC')
        elif contador == 2:
            new_cols.append('FECHA_INICIO_TNG')
        elif contador == 3:
            new_cols.append('FECHA_INICIO_PRO')
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'RESUELVE_{contador}')
    else:
        new_cols.append(col_str)

df_excel_distancia.columns = new_cols

In [12]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_distancia.columns:
    col_str = str(col)

    if col_str.startswith('FECHA SNIES VENCIMIENTO'):
        contador += 1
        if contador == 1:
            new_cols.append('FECHA_FIN_TEC')
        elif contador == 2:
            new_cols.append('FECHA_FIN_TNG')
        elif contador == 3:
            new_cols.append('FECHA_FIN_PRO')
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'FECHA SNIES VENCIMIENTO_{contador}')
    else:
        new_cols.append(col_str)

df_excel_distancia.columns = new_cols

In [13]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_distancia.columns:
    col_str = str(col)

    if col_str.startswith('NOMBRE_DEL_PROGRAMA'):
        contador += 1
        if contador == 1:
            new_cols.append('NOMBRE_DEL_PROGRAMA_TEC')
        elif contador == 2:
            new_cols.append('NOMBRE_DEL_PROGRAMA_TNG')
        elif contador == 3:
            new_cols.append('NOMBRE_DEL_PROGRAMA_PRO')
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'NOMBRE_DEL_PROGRAMA_{contador}')
    else:
        new_cols.append(col_str)

df_excel_distancia.columns = new_cols

In [14]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_distancia.columns:
    col_str = str(col)

    if col_str.startswith('TITULO_OTORGADO'):
        contador += 1
        if contador == 1:
            new_cols.append('TITULO_OTORGADO_TEC')
        elif contador == 2:
            new_cols.append('TITULO_OTORGADO_TNG')
        elif contador == 3:
            new_cols.append('TITULO_OTORGADO_PRO')
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'TITULO_OTORGADO_{contador}')
    else:
        new_cols.append(col_str)

df_excel_distancia.columns = new_cols

In [15]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_distancia.columns:
    col_str = str(col)

    if col_str.startswith('TOTAL CREDITOS'):
        contador += 1
        if contador == 1:
            new_cols.append('TOTAL_CREDITOS_TECNICO')
        elif contador == 2:
            new_cols.append('TOTAL_CREDITOS_TECNOLOGO')
        elif contador == 3:
            new_cols.append('TOTAL_CREDITOS_PROFESIONAL')   
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'TOTAL CREDITOS_{contador}')
    else:
        new_cols.append(col_str)

df_excel_distancia.columns = new_cols

In [16]:
#se renombran las columnas
df_excel_distancia = df_excel_distancia.rename(columns={'PROGRAMA':'PROGRAMA_PROFESIONAL','TECNICO' : 'COD_PROG_TECNICO',
                                    'CODIGO SNIES TECNICO':'CODIGO_SNIES_TECNICO','TECNOLOGO':'COD_PROG_TECNOLOGO',
                                    'CODIGO SNIES TECNOLOGICO':'CODIGO_SNIES_TECNOLOGICO','PROFESIONAL':'COD_PROG_PROFESIONAL',
                                    'CODIGO SNIES PROFESIONAL':'CODIGO_SNIES_PROF','TECNICO NIVELES POR CICLO':'TECNICO_NIVELES',
                                    'TECNOLOGICO NIVELES POR CICLO':'TECNOLOGO_NIVELES','PROFESIONAL NIVELES POR CICLO':'PROFESIONAL_NIVELES','TOTAL CREDITO MALLA':'TOTAL_CRED_PROF'
                                    })

In [17]:
#se crean columnas que no existen en el archivo
df_excel_distancia['11C'] = 'N/A'

In [18]:
#se eliminan vacios 
df_excel_distancia.columns = df_excel_distancia.columns.str.replace(' ', '')

In [19]:
# definicion de columnas
df_mallas_laura_distancia = df_excel_distancia[['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'PENSUM', 'COD_PROG_TECNICO', 'RESOLUCION_TECNICO', 'FECHA_INICIO_TEC',
       'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC', 'NOMBRE_DEL_PROGRAMA_TEC',
       'TITULO_OTORGADO_TEC', 'COD_PROG_TECNOLOGO', 'RESOLUCION_TNG',
       'FECHA_INICIO_TNG', 'CODIGO_SNIES_TECNOLOGICO', 'FECHA_FIN_TNG',
       'NOMBRE_DEL_PROGRAMA_TNG', 'TITULO_OTORGADO_TNG',
       'COD_PROG_PROFESIONAL', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', 'TECNOLOGO_NIVELES',
       'TOTAL_CREDITOS_TECNOLOGO', 'PROFESIONAL_NIVELES',
       'TOTAL_CREDITOS_PROFESIONAL', 'TOTAL_CRED_PROF', '1C', '2C', '3C', '4C',
       '5C', '6C', '7C', '8C', '9C', '10C', '11C']].copy()

In [20]:
#transformacion de fechas
df_mallas_laura_distancia['FECHA_INICIO_TEC'] = pd.to_datetime(df_mallas_laura_distancia['FECHA_INICIO_TEC'], errors='coerce')
df_mallas_laura_distancia['FECHA_FIN_TEC'] = pd.to_datetime(df_mallas_laura_distancia['FECHA_FIN_TEC'], errors='coerce')
df_mallas_laura_distancia['FECHA_INICIO_TNG'] = pd.to_datetime(df_mallas_laura_distancia['FECHA_INICIO_TNG'], errors='coerce')
df_mallas_laura_distancia['FECHA_FIN_TNG'] = pd.to_datetime(df_mallas_laura_distancia['FECHA_FIN_TNG'], errors='coerce')
df_mallas_laura_distancia['FECHA_INICIO_PRO'] = pd.to_datetime(df_mallas_laura_distancia['FECHA_INICIO_PRO'], errors='coerce')
df_mallas_laura_distancia['FECHA_FIN_PRO'] = pd.to_datetime(df_mallas_laura_distancia['FECHA_FIN_PRO'], errors='coerce')


df_mallas_laura_distancia['FECHA_INICIO_TEC'] = df_mallas_laura_distancia['FECHA_INICIO_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_distancia['FECHA_FIN_TEC'] = df_mallas_laura_distancia['FECHA_FIN_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_distancia['FECHA_INICIO_TNG'] = df_mallas_laura_distancia['FECHA_INICIO_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_distancia['FECHA_FIN_TNG'] = df_mallas_laura_distancia['FECHA_FIN_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_distancia['FECHA_INICIO_PRO'] = df_mallas_laura_distancia['FECHA_INICIO_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_distancia['FECHA_FIN_PRO'] = df_mallas_laura_distancia['FECHA_FIN_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')

In [21]:
df_excel_especializacion = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\ESTUDIANTES_V2\MALLAS\ARCHIVOS_MALLAS\MALLAS CUN ACTUALIZADO_JUNIO_2026.xlsx', sheet_name='ESPECIALIZACION')
df_excel_especializacion

,ESTADO,ESCUELA,PROGRAMA,SEDE,MODALIDAD,POSGRADO,PENSUM,RESOLUCION,RESUELVE,CODIGO SNIES,VIGENCIA ESPECIALIZACION,NOMBRE_DEL_PROGRAMA,TITULO_OTORGADO,AÑO DE REGISTRO,IDIOMAS,1 C,2 C,3 C,4 C,TOTAL CRED PROGRAMA
0,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,BOGOTA CENTRO,VIRTUAL,ESP08,ESPPA,021993,2016-11-22,106040,2023-11-21,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,ESPECIALISTA EN PAZ Y DESARROLLO TERRITORIAL,2016,NaN,7,5,10.0,4.0,26
1,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,ESPECIALIZACION EN GERENCIA DE LA INDUSTRIA EN...,BOGOTA CENTRO,VIRTUAL,ESP09,ESPMO,16895,2016-08-22,105784,2023-08-21,ESPECIALIZACION EN GERENCIA EN LA INDUSTRIA DE...,ESPECIALISTA EN GERENCIA EN LA INDUSTRIA DE LA...,2016,NaN,8,8,10.0,NaN,26
2,VIGENTE,ESCUELA DE INGENIERIAS,ESPECIALIZACION EN ANALITICA DE DATOS,BOGOTA CENTRO,VIRTUAL,ESP11,ESPAD,021976,2020-11-24,109601,2027-11-23,ESPECIALIZACION EN ANALITICA DE DATOS,ESPECIALISTA EN ANALITICA DE DATOS,2020,NaN,6,7,8.0,7.0,28
3,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESPECIALIZACION EN GERENCIA DE LA MARCA,BOGOTA CENTRO,VIRTUAL,ESP10,ESPMA,011893,2019-11-14,108460,2026-11-12,ESPECIALIZACION EN GERENCIA DE LA MARCA,ESPECIALISTA EN GERENCIA DE LA MARCA,2019,NaN,6,8,6.0,6.0,26
4,VIGENTE,ESCUELA DE INGENIERIAS,ESPECIALIZACION EN TRANSFORMACION DIGITAL,BOGOTA CENTRO,VIRTUAL,ESP12,ESPTD,025083,2021-12-29,110990,2028-12-27,ESPECIALIZACION EN TRANSFORMACION DIGITAL,ESPECIALISTA EN TRANSFORMACION DIGITAL,2022,NaN,6,7,8.0,7.0,28
5,VIGENTE,ESCUELA DE INGENIERIAS,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,BOGOTA CENTRO,VIRTUAL,ESP13,ESPTI,10711,2023-07-07,111621,2030-07-05,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,ESPECIALISTA EN GESTION DE TECNOLOGIAS DE INFO...,2023,NaN,6,7,8.0,7.0,28
6,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESPECIALIZACION EN GESTION DE NEGOCIOS DIGITALES,BOGOTA CENTRO,VIRTUAL,ESP14,ESPGN,025193,2024-12-24,117624,2031-12-23,ESPECIALIZACION EN GESTION DE NEGOCIOS DIGITALES,ESPECIALISTA EN GESTION DE NEGOCIOS DIGITALES,2024,NaN,14,14,NaN,NaN,28
7,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,BOGOTA CENTRO,VIRTUAL,ESP15,ESPGE,025188,2024-12-24,117591,2031-12-23,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,ESPECIALISTA EN GERENCIA EDUCATIVA,2024,NaN,14,14,NaN,NaN,28
8,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,BOGOTA CENTRO,VIRTUAL,ESP16,ESPDT,001279,2025-01-31,117723,2032-01-30,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,ESPECIALISTA EN DESARROLLO ORGANIZACIONAL Y GE...,2025,NaN,13,15,NaN,NaN,28
9,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA,BOGOTA CENTRO,VIRTUAL,ESP17,ESPPE,001807,2025-02-10,117816,2032-02-09,ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA,ESPECIALISTA EN PROSPECTIVA ESTRATÉGICA,2025,NaN,14,14,NaN,NaN,28


In [22]:
# se renombran algunas columnas
df_excel_especializacion = df_excel_especializacion.rename(columns={'ESCUELA':'ESCUELAS','PROGRAMA ':'PROGRAMA_PROFESIONAL','POSGRADO':'COD_PROG_PROFESIONAL',
                                                        'RESOLUCION':'RESOLUCION_PROF','RESUELVE':'FECHA_INICIO_PRO','CODIGO SNIES ':'CODIGO_SNIES_PROF',
                                                        'VIGENCIA ESPECIALIZACION':'FECHA_FIN_PRO','NOMBRE_DEL_PROGRAMA':'NOMBRE_DEL_PROGRAMA_PRO',
                                                        'TITULO_OTORGADO':'TITULO_OTORGADO_PRO','TOTAL CRED PROGRAMA':'TOTAL_CRED_PROF'
                                    })

In [23]:
df_excel_especializacion.columns

Index(['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'COD_PROG_PROFESIONAL', 'PENSUM', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'AÑO DE REGISTRO', 'IDIOMAS', '1 C ', '2 C ',
       '3 C ', '4 C ', 'TOTAL_CRED_PROF'],
      dtype='object')

In [24]:
# se crean columnas que no vienen en el archivo
df_excel_especializacion['TOTAL_CREDITOS_PROFESIONAL'] = df_excel_especializacion['TOTAL_CRED_PROF'] 
df_excel_especializacion['5C'] = 'N/A'
df_excel_especializacion['6C'] = 'N/A'
df_excel_especializacion['7C'] = 'N/A'
df_excel_especializacion['8C'] = 'N/A'
df_excel_especializacion['9C'] = 'N/A'
df_excel_especializacion['10C'] = 'N/A'
df_excel_especializacion['11C'] = 'N/A'

In [25]:
# se crean columnas que no vienen en el archivo
df_excel_especializacion['COD_PROG_TECNICO'] = 'N/A'
df_excel_especializacion['RESOLUCION_TECNICO'] = 'N/A'
df_excel_especializacion['FECHA_INICIO_TEC'] = 'N/A'
df_excel_especializacion['CODIGO_SNIES_TECNICO'] = 'N/A'
df_excel_especializacion['FECHA_FIN_TEC'] = 'N/A'
df_excel_especializacion['NOMBRE_DEL_PROGRAMA_TEC'] = 'N/A'
df_excel_especializacion['TITULO_OTORGADO_TEC'] = 'N/A'
df_excel_especializacion['COD_PROG_TECNOLOGO'] = 'N/A'
df_excel_especializacion['RESOLUCION_TNG'] = 'N/A'
df_excel_especializacion['FECHA_INICIO_TNG'] = 'N/A'
df_excel_especializacion['CODIGO_SNIES_TECNOLOGICO'] = 'N/A'
df_excel_especializacion['FECHA_FIN_TNG'] = 'N/A'
df_excel_especializacion['NOMBRE_DEL_PROGRAMA_TNG'] = 'N/A'
df_excel_especializacion['TITULO_OTORGADO_TNG'] = 'N/A'
df_excel_especializacion['TECNICO_NIVELES'] = 'N/A'
df_excel_especializacion['TOTAL_CREDITOS_TECNICO'] = 'N/A'
df_excel_especializacion['TECNOLOGO_NIVELES'] = 'N/A'
df_excel_especializacion['TOTAL_CREDITOS_TECNOLOGO'] = 'N/A'
df_excel_especializacion['PROFESIONAL_NIVELES'] = 'N/A'

In [26]:
#se eliminan vacios
df_excel_especializacion.columns = df_excel_especializacion.columns.str.replace(' ', '')

In [27]:
# definicion de columnas
df_mallas_laura_especializacion = df_excel_especializacion[['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'PENSUM', 'COD_PROG_TECNICO', 'RESOLUCION_TECNICO', 'FECHA_INICIO_TEC',
       'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC', 'NOMBRE_DEL_PROGRAMA_TEC',
       'TITULO_OTORGADO_TEC', 'COD_PROG_TECNOLOGO', 'RESOLUCION_TNG',
       'FECHA_INICIO_TNG', 'CODIGO_SNIES_TECNOLOGICO', 'FECHA_FIN_TNG',
       'NOMBRE_DEL_PROGRAMA_TNG', 'TITULO_OTORGADO_TNG',
       'COD_PROG_PROFESIONAL', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', 'TECNOLOGO_NIVELES',
       'TOTAL_CREDITOS_TECNOLOGO', 'PROFESIONAL_NIVELES',
       'TOTAL_CREDITOS_PROFESIONAL', 'TOTAL_CRED_PROF', '1C', '2C', '3C', '4C',
       '5C', '6C', '7C', '8C', '9C', '10C', '11C']].copy()

In [28]:
#transformacion de fechas
df_mallas_laura_especializacion['FECHA_INICIO_TEC'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_INICIO_TEC'], errors='coerce')
df_mallas_laura_especializacion['FECHA_FIN_TEC'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_FIN_TEC'], errors='coerce')
df_mallas_laura_especializacion['FECHA_INICIO_TNG'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_INICIO_TNG'], errors='coerce')
df_mallas_laura_especializacion['FECHA_FIN_TNG'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_FIN_TNG'], errors='coerce')
df_mallas_laura_especializacion['FECHA_INICIO_PRO'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_INICIO_PRO'], errors='coerce')
df_mallas_laura_especializacion['FECHA_FIN_PRO'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_FIN_PRO'], errors='coerce')


df_mallas_laura_especializacion['FECHA_INICIO_TEC'] = df_mallas_laura_especializacion['FECHA_INICIO_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_especializacion['FECHA_FIN_TEC'] = df_mallas_laura_especializacion['FECHA_FIN_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_especializacion['FECHA_INICIO_TNG'] = df_mallas_laura_especializacion['FECHA_INICIO_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_especializacion['FECHA_FIN_TNG'] = df_mallas_laura_especializacion['FECHA_FIN_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_especializacion['FECHA_INICIO_PRO'] = df_mallas_laura_especializacion['FECHA_INICIO_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_especializacion['FECHA_FIN_PRO'] = df_mallas_laura_especializacion['FECHA_FIN_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2084\919895683.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_mallas_laura_especializacion['FECHA_INICIO_TEC'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_INICIO_TEC'], errors='coerce')
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2084\919895683.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_mallas_laura_especializacion['FECHA_FIN_TEC'] = pd.to_datetime(df_mallas_laura_especializacion['FECHA_FIN_TEC'], errors='coerce')
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2084\919895683.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure

In [29]:
df_excel_articulacion = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\ESTUDIANTES_V2\MALLAS\ARCHIVOS_MALLAS\MALLAS CUN ACTUALIZADO_JUNIO_2026.xlsx', sheet_name='ARTICULACION')
df_excel_articulacion

,ESTADO,ESCUELAS,PROGRAMA,SEDE,MODALIDAD,TECNICO,PEMSUN,RESOLUCION,RESUELVE,COdigo SNIES TECNICO,FECHA SNIES,NOMBRE_DEL_PROGRAMA,TITULO_OTORGADO,IDIOMAS,TECNICO NIVELES POR CICLO,TOTAL CREDITOS,1 sem-cred,2 sem-cred,3 sem-cred,4 sem-cred
0,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,SANTA MARTA,PRESENCIAL,2TA01,1030,21274,2014-12-16,6101,2021-12-14,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,FUERA,3,55,18,18,19,NaN
1,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,SANTA MARTA,PRESENCIAL,2TA02,1030,21274,2014-12-16,6101,2021-12-14,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,FUERA,3,55,18,18,19,NaN
2,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,SANTA MARTA,PRESENCIAL,2TA22,1030,21274,2014-12-16,6101,2021-12-14,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,FUERA,3,55,18,18,19,NaN
3,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,SANTA MARTA,PRESENCIAL,1TA75,1030,21274,2014-12-16,6101,2021-12-14,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,FUERA,3,55,18,18,19,NaN
4,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,SANTA MARTA,PRESENCIAL,2TA29,1030,21274,2014-12-16,6101,2021-12-14,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVOS,FUERA,3,55,18,18,19,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,VIGENTE,ESCUELA DE DISEÑO Y COMUNICACION,TECNICA PROFESIONAL EN OPERACION DE PROCESOS D...,BOGOTA,VIRTUAL,4TAGA,5016,´00849,2024-02-08,111728,2031-02-06,TECNICA PROFESIONAL EN OPERACION DE PROCESOS D...,TECNICO/A PROFESIONAL EN OPERACION DE PROCESOS...,FUERA,4,67,17,16,17,17.0
361,VIGENTE,ESCUELA DE DISEÑO Y COMUNICACION,TECNICA PROFESIONAL EN OPERACION DE PROCESOS D...,BOGOTA,VIRTUAL,4TABG,5016V,´00849,2024-02-08,111729,2031-02-06,TECNICA PROFESIONAL EN OPERACION DE PROCESOS D...,TECNICO/A PROFESIONAL EN OPERACION DE PROCESOS...,FUERA,4,67,17,16,17,17.0
362,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS AGROINDUSTRIALES,VIRTUAL,VIRTUAL,3TA84,1045V,01514,2024-02-21,105504,2031-02-19,TÉCNICA PROFESIONAL EN PRODUCCIÓN DE MATERIAS ...,TÉCNICO/A PROFESIONAL EN PRODUCCIÓN DE MATERIA...,FUERA,3,54,18,18,18,NaN
363,VIGENTE,"ESCUELA DE CIENCIAS SOCIALES, JURIDICAS Y GOBI...",TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,VIRTUAL,VIRTUAL,3TA85,VTG12,12003,2023-07-14,111712,2030-07-12,TECNICA PROFESIONAL EN OPERACION DE PROCESOS A...,TECNICO PROFESIONAL EN OPERACION DE PROCESOS A...,FUERA,4,67,17,17,16,17.0


In [30]:
df_excel_articulacion = df_excel_articulacion.rename(columns={'PROGRAMA':'PROGRAMA_PROFESIONAL','PEMSUN':'PENSUM','TECNICO':'COD_PROG_TECNICO',
                                                        'RESOLUCION':'RESOLUCION_TECNICO','RESUELVE':'FECHA_INICIO_TEC','COdigo SNIES TECNICO':'CODIGO_SNIES_TECNICO',
                                                        'FECHA SNIES':'FECHA_FIN_TEC','NOMBRE_DEL_PROGRAMA':'NOMBRE_DEL_PROGRAMA_TEC',
                                                        'TITULO_OTORGADO':'TITULO_OTORGADO_TEC','TECNICO NIVELES POR CICLO':'TECNICO_NIVELES',
                                                        'TOTAL CREDITOS':'TOTAL_CREDITOS_TECNICO','1 sem-cred':'1C','2 sem-cred':'2C','3 sem-cred':'3C',
                                                        '4 sem-cred':'4C'})

In [31]:
df_excel_articulacion.columns

Index(['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'COD_PROG_TECNICO', 'PENSUM', 'RESOLUCION_TECNICO', 'FECHA_INICIO_TEC',
       'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC', 'NOMBRE_DEL_PROGRAMA_TEC',
       'TITULO_OTORGADO_TEC', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', '1C', '2C', '3C', '4C'],
      dtype='object')

In [32]:
# se crean columnas que no vienen en el archivo
df_excel_articulacion['TOTAL_CRED_PROF'] = df_excel_articulacion['TOTAL_CREDITOS_TECNICO'] 
df_excel_articulacion['5C'] = 'N/A'
df_excel_articulacion['6C'] = 'N/A'
df_excel_articulacion['7C'] = 'N/A'
df_excel_articulacion['8C'] = 'N/A'
df_excel_articulacion['9C'] = 'N/A'
df_excel_articulacion['10C'] = 'N/A'
df_excel_articulacion['11C'] = 'N/A'

In [33]:
# se crean columnas que no vienen en el archivo
df_excel_articulacion['COD_PROG_TECNOLOGO'] = 'N/A'
df_excel_articulacion['RESOLUCION_TNG'] = 'N/A'
df_excel_articulacion['FECHA_INICIO_TNG'] = 'N/A'
df_excel_articulacion['CODIGO_SNIES_TECNOLOGICO'] = 'N/A'
df_excel_articulacion['FECHA_FIN_TNG'] = 'N/A'
df_excel_articulacion['NOMBRE_DEL_PROGRAMA_TNG'] = 'N/A'
df_excel_articulacion['TITULO_OTORGADO_TNG'] = 'N/A'
df_excel_articulacion['COD_PROG_PROFESIONAL'] = 'N/A'
df_excel_articulacion['RESOLUCION_PROF'] = 'N/A'
df_excel_articulacion['FECHA_INICIO_PRO'] = 'N/A'
df_excel_articulacion['CODIGO_SNIES_PROF'] = 'N/A'
df_excel_articulacion['FECHA_FIN_PRO'] = 'N/A'
df_excel_articulacion['NOMBRE_DEL_PROGRAMA_PRO'] = 'N/A'
df_excel_articulacion['TITULO_OTORGADO_PRO'] = 'N/A'
df_excel_articulacion['TECNOLOGO_NIVELES'] = 'N/A'
df_excel_articulacion['TOTAL_CREDITOS_TECNOLOGO'] = 'N/A'
df_excel_articulacion['PROFESIONAL_NIVELES'] = 'N/A'
df_excel_articulacion['TOTAL_CREDITOS_PROFESIONAL'] = 'N/A'

In [34]:
# definicion de columnas
df_mallas_laura_articulacion = df_excel_articulacion[['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'PENSUM', 'COD_PROG_TECNICO', 'RESOLUCION_TECNICO', 'FECHA_INICIO_TEC',
       'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC', 'NOMBRE_DEL_PROGRAMA_TEC',
       'TITULO_OTORGADO_TEC', 'COD_PROG_TECNOLOGO', 'RESOLUCION_TNG',
       'FECHA_INICIO_TNG', 'CODIGO_SNIES_TECNOLOGICO', 'FECHA_FIN_TNG',
       'NOMBRE_DEL_PROGRAMA_TNG', 'TITULO_OTORGADO_TNG',
       'COD_PROG_PROFESIONAL', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', 'TECNOLOGO_NIVELES',
       'TOTAL_CREDITOS_TECNOLOGO', 'PROFESIONAL_NIVELES',
       'TOTAL_CREDITOS_PROFESIONAL', 'TOTAL_CRED_PROF', '1C', '2C', '3C', '4C',
       '5C', '6C', '7C', '8C', '9C', '10C', '11C']].copy()

In [35]:
#transformacion de fechas
df_mallas_laura_articulacion['FECHA_INICIO_TEC'] = pd.to_datetime(df_mallas_laura_articulacion['FECHA_INICIO_TEC'], errors='coerce')
df_mallas_laura_articulacion['FECHA_FIN_TEC'] = pd.to_datetime(df_mallas_laura_articulacion['FECHA_FIN_TEC'], errors='coerce')
df_mallas_laura_articulacion['FECHA_INICIO_TNG'] = pd.to_datetime(df_mallas_laura_articulacion['FECHA_INICIO_TNG'], errors='coerce')
df_mallas_laura_articulacion['FECHA_FIN_TNG'] = pd.to_datetime(df_mallas_laura_articulacion['FECHA_FIN_TNG'], errors='coerce')
df_mallas_laura_articulacion['FECHA_INICIO_PRO'] = pd.to_datetime(df_mallas_laura_articulacion['FECHA_INICIO_PRO'], errors='coerce')
df_mallas_laura_articulacion['FECHA_FIN_PRO'] = pd.to_datetime(df_mallas_laura_articulacion['FECHA_FIN_PRO'], errors='coerce')


df_mallas_laura_articulacion['FECHA_INICIO_TEC'] = df_mallas_laura_articulacion['FECHA_INICIO_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_articulacion['FECHA_FIN_TEC'] = df_mallas_laura_articulacion['FECHA_FIN_TEC'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_articulacion['FECHA_INICIO_TNG'] = df_mallas_laura_articulacion['FECHA_INICIO_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_articulacion['FECHA_FIN_TNG'] = df_mallas_laura_articulacion['FECHA_FIN_TNG'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_articulacion['FECHA_INICIO_PRO'] = df_mallas_laura_articulacion['FECHA_INICIO_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
df_mallas_laura_articulacion['FECHA_FIN_PRO'] = df_mallas_laura_articulacion['FECHA_FIN_PRO'].dt.strftime('%Y-%m-%d %H:%M:%S.000')

In [36]:
df_excel_fds = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\ESTUDIANTES_V2\MALLAS\ARCHIVOS_MALLAS\MALLAS CUN ACTUALIZADO_JUNIO_2026.xlsx', sheet_name='PRESENCIAL FINES DE SEMANA')
df_excel_fds

,ESTADO,ESCUELAS,PROGRAMA,SEDE,MODALIDAD,PENSUM,COD PROGR TECNICO,RESOLUCION,RESUELVE,COdigo SNIES TECNICO,...,3 C,4 C,5 C,6 C,7 C,8 C,9 C,10 C,11,TOTAL CRED MALLA
0,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN OPERACION DE ALIMENTOS ...,BOGOTA FONTIBON,PRESENCIAL,8035,PT838,6893,2010-08-06,54863,...,16,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66
1,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION TURISTICA Y HOTELERA,BOGOTA FONTIBON,PRESENCIAL,8036,PT846,4529,2014-04-01,2381,...,22,19,25.0,16.0,16.0,19.0,16.0,NaN,NaN,168
2,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN OPERACION DE ALIMENTOS ...,BOGOTA NORTE,PRESENCIAL,8035,PT181,6893,2010-08-06,54863,...,16,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66
3,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION TURISTICA Y HOTELERA,BOGOTA NORTE,PRESENCIAL,8036,PT136,4529,2014-04-01,2381,...,22,19,25.0,16.0,16.0,19.0,16.0,NaN,NaN,168
4,VECIDO,ESCUELA DE DISEÑO Y COMUNICACION,DISEÑO GRAFICO,BOGOTA NORTE,PRESENCIAL,5015,PT133,10506,2015-07-14,104744,...,20,20,20.0,19.0,20.0,19.0,21.0,NaN,NaN,178
5,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN OPERACION DE ALIMENTOS ...,BOGOTA RESTREPO,PRESENCIAL,8035,PT607,6893,2010-08-06,54863,...,16,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66
6,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION TURISTICA Y HOTELERA,BOGOTA RESTREPO,PRESENCIAL,8036,PT400,4529,2014-04-01,2381,...,22,19,25.0,16.0,16.0,19.0,16.0,NaN,NaN,168
7,VECIDO,ESCUELA DE DISEÑO Y COMUNICACION,DISEÑO GRAFICO,BOGOTA RESTREPO,PRESENCIAL,5015,PT602,10506,2015-07-14,104744,...,20,20,20.0,19.0,20.0,19.0,21.0,NaN,NaN,178
8,VECIDO,ESCUELA DE INGENIERIAS,INGENIERIA ELECTRONICA,BOGOTA SUBA,PRESENCIAL,2031,PT511,´087,2014-01-03,2384,...,17,17,21.0,20.0,20.0,20.0,21.0,20.0,NaN,194
9,VECIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN OPERACION DE ALIMENTOS ...,BOGOTA SUBA,PRESENCIAL,8035,PT510,6893,2010-08-06,54863,...,16,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66


In [37]:
df_excel_fds.columns

Index([                       'ESTADO',                      'ESCUELAS',
                            'PROGRAMA',                          'SEDE',
                           'MODALIDAD',                        'PENSUM',
                   'COD PROGR TECNICO',                    'RESOLUCION',
                            'RESUELVE',          'COdigo SNIES TECNICO',
             'FECHA SNIES VENCIMIENTO',           'NOMBRE_DEL_PROGRAMA',
                     'TITULO_OTORGADO',                       'IDIOMAS',
           'TECNICO NIVELES POR CICLO',                'TOTAL CREDITOS',
         'TECNOLOGO NIVELES POR CICLO',              'TOTAL CREDITOS.1',
       'PROFESIONAL NIVELES POR CICLO',              'TOTAL CREDITOS.2',
                                '1 C ',                          '2 C ',
                                '3 C ',                          '4 C ',
                                '5 C ',                          '6 C ',
                                '7 C ',            

In [38]:
# se renombran algunas columnas
df_excel_fds = df_excel_fds.rename(columns={'PROGRAMA':'PROGRAMA_PROFESIONAL','COD PROGR TECNICO':'COD_PROG_TECNICO',
                                            'RESOLUCION':'RESOLUCION_TECNICO','RESUELVE':'FECHA_INICIO_TEC','COdigo SNIES TECNICO':'CODIGO_SNIES_TECNICO',
                                            'FECHA SNIES VENCIMIENTO':'FECHA_FIN_TEC','NOMBRE_DEL_PROGRAMA':'NOMBRE_DEL_PROGRAMA_TEC',
                                            'TITULO_OTORGADO':'TITULO_OTORGADO_TEC','TECNICO NIVELES POR CICLO':'TECNICO_NIVELES',
                                            'TECNOLOGO NIVELES POR CICLO':'TECNOLOGO_NIVELES','TOTAL CRED MALLA':'TOTAL_CRED_PROF',
                                            11:'11C'})

In [39]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df_excel_fds.columns:
    col_str = str(col)

    if col_str.startswith('TOTAL CREDITOS'):
        contador += 1
        if contador == 1:
            new_cols.append('TOTAL_CREDITOS_TECNICO')
        elif contador == 2:
            new_cols.append('TOTAL_CREDITOS_TECNOLOGO')
        elif contador == 3:
            new_cols.append('TOTAL_CREDITOS_PROFESIONAL')   
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'TOTAL CREDITOS_{contador}')
    else:
        new_cols.append(col_str)

df_excel_fds.columns = new_cols

In [40]:
# se eliminan vacios
df_excel_fds.columns = df_excel_fds.columns.str.replace(' ', '')

In [41]:
# se crean columnas que no vienen en el archivo
df_excel_fds['COD_PROG_TECNOLOGO'] = 'N/A'
df_excel_fds['RESOLUCION_TNG'] = 'N/A'
df_excel_fds['FECHA_INICIO_TNG'] = 'N/A'
df_excel_fds['CODIGO_SNIES_TECNOLOGICO'] = 'N/A'
df_excel_fds['FECHA_FIN_TNG'] = 'N/A'
df_excel_fds['NOMBRE_DEL_PROGRAMA_TNG'] = 'N/A'
df_excel_fds['TITULO_OTORGADO_TNG'] = 'N/A'
df_excel_fds['COD_PROG_PROFESIONAL'] = 'N/A'
df_excel_fds['RESOLUCION_PROF'] = 'N/A'
df_excel_fds['FECHA_INICIO_PRO'] = 'N/A'
df_excel_fds['CODIGO_SNIES_PROF'] = 'N/A'
df_excel_fds['FECHA_FIN_PRO'] = 'N/A'
df_excel_fds['NOMBRE_DEL_PROGRAMA_PRO'] = 'N/A'
df_excel_fds['TITULO_OTORGADO_PRO'] = 'N/A'
df_excel_fds['PROFESIONAL_NIVELES'] = 'N/A'


In [42]:
# definicion de columnas
df_mallas_laura_fds = df_excel_fds[['ESTADO', 'ESCUELAS', 'PROGRAMA_PROFESIONAL', 'SEDE', 'MODALIDAD',
       'PENSUM', 'COD_PROG_TECNICO', 'RESOLUCION_TECNICO', 'FECHA_INICIO_TEC',
       'CODIGO_SNIES_TECNICO', 'FECHA_FIN_TEC', 'NOMBRE_DEL_PROGRAMA_TEC',
       'TITULO_OTORGADO_TEC', 'COD_PROG_TECNOLOGO', 'RESOLUCION_TNG',
       'FECHA_INICIO_TNG', 'CODIGO_SNIES_TECNOLOGICO', 'FECHA_FIN_TNG',
       'NOMBRE_DEL_PROGRAMA_TNG', 'TITULO_OTORGADO_TNG',
       'COD_PROG_PROFESIONAL', 'RESOLUCION_PROF', 'FECHA_INICIO_PRO',
       'CODIGO_SNIES_PROF', 'FECHA_FIN_PRO', 'NOMBRE_DEL_PROGRAMA_PRO',
       'TITULO_OTORGADO_PRO', 'IDIOMAS', 'TECNICO_NIVELES',
       'TOTAL_CREDITOS_TECNICO', 'TECNOLOGO_NIVELES',
       'TOTAL_CREDITOS_TECNOLOGO', 'PROFESIONAL_NIVELES',
       'TOTAL_CREDITOS_PROFESIONAL', 'TOTAL_CRED_PROF', '1C', '2C', '3C', '4C',
       '5C', '6C', '7C', '8C', '9C', '10C', '11C']].copy()

In [43]:
df_final_pre = pd.concat([df_mallas_laura,df_mallas_laura_distancia,df_mallas_laura_especializacion,df_mallas_laura_articulacion,df_mallas_laura_fds])
df_final_pre.shape

(1728, 46)

In [44]:
# se normalizan los estados
df_final_pre.loc[df_final_pre['ESTADO'].str.contains('VE', case=False, na=False),'ESTADO'] = 'VENCIDO'
df_final_pre.loc[df_final_pre['ESTADO'].str.contains('VI', case=False, na=False),'ESTADO'] = 'VIGENTE'

In [45]:
# eliminacion de caracteres
df_final_pre['RESOLUCION_TECNICO'] = (
    df_final_pre['RESOLUCION_TECNICO']
    .astype(str)
    .str.replace(r"[\'´`]", "", regex=True)  # elimina varias comillas
    .str.strip()
)

df_final_pre['RESOLUCION_TNG'] = (
    df_final_pre['RESOLUCION_TNG']
    .astype(str)
    .str.replace(r"[\'´`]", "", regex=True)  # elimina varias comillas
    .str.strip()
)
    
df_final_pre['RESOLUCION_PROF'] = (
    df_final_pre['RESOLUCION_PROF'
    '']
    .astype(str)
    .str.replace(r"[\'´`]", "", regex=True)  # elimina varias comillas
    .str.strip()
)


In [46]:
df_final_pre

,ESTADO,ESCUELAS,PROGRAMA_PROFESIONAL,SEDE,MODALIDAD,PENSUM,COD_PROG_TECNICO,RESOLUCION_TECNICO,FECHA_INICIO_TEC,CODIGO_SNIES_TECNICO,...,2C,3C,4C,5C,6C,7C,8C,9C,10C,11C
0,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,COMUNICACION SOCIAL,BARRANQUILLA,PRESENCIAL,3031,1TQ12,16932,2016-08-22 00:00:00.000,105792,...,16.0,19.0,17.0,19.0,19.0,16.0,17.0,17.0,NaN,NaN
1,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3024,1TC11,3567,2009-06-01 00:00:00.000,54658,...,18.0,19.0,20.0,18.0,19.0,18.0,20.0,18.0,20.0,NaN
2,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3027,1TC11,3567,2009-06-01 00:00:00.000,54658,...,20.0,19.0,20.0,18.0,21.0,21.0,18.0,19.0,18.0,NaN
3,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3028,1TC11,10711,2016-06-01 00:00:00.000,54658,...,18.0,18.0,20.0,20.0,20.0,19.0,17.0,18.0,NaN,NaN
4,VENCIDO,ESCUELA DE INGENIERIAS,INGENIERIA DE SISTEMAS,BOGOTA CENTRO,PRESENCIAL,2011,1TC03,421,2011-02-06 00:00:00.000,2383,...,21.0,20.0,19.0,20.0,21.0,20.0,20.0,20.0,19.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA RESTREPO,PRESENCIAL,1034,PT401,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
43,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA SUBA,PRESENCIAL,1034,PT501,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
44,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,NEIVA,PRESENCIAL,1030,PT091,13435,2014-08-19 00:00:00,6124,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
45,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN SERVICIOS ADMINISTRATIVOS,MONTERIA,PRESENCIAL,6015,PT205,14426,2015-09-07 00:00:00,54274,...,17.0,20.0,20.0,20.0,15.0,17.0,20.0,17.0,NaN,NaN


In [47]:
df_final = df_final_pre.copy()

In [48]:
df_final = df_final.assign(
    PENSUM=df_final['PENSUM'].astype(str).str.split(r'[/,;]')
).explode('PENSUM')
df_final

,ESTADO,ESCUELAS,PROGRAMA_PROFESIONAL,SEDE,MODALIDAD,PENSUM,COD_PROG_TECNICO,RESOLUCION_TECNICO,FECHA_INICIO_TEC,CODIGO_SNIES_TECNICO,...,2C,3C,4C,5C,6C,7C,8C,9C,10C,11C
0,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,COMUNICACION SOCIAL,BARRANQUILLA,PRESENCIAL,3031,1TQ12,16932,2016-08-22 00:00:00.000,105792,...,16.0,19.0,17.0,19.0,19.0,16.0,17.0,17.0,NaN,NaN
1,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3024,1TC11,3567,2009-06-01 00:00:00.000,54658,...,18.0,19.0,20.0,18.0,19.0,18.0,20.0,18.0,20.0,NaN
2,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3027,1TC11,3567,2009-06-01 00:00:00.000,54658,...,20.0,19.0,20.0,18.0,21.0,21.0,18.0,19.0,18.0,NaN
3,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3028,1TC11,10711,2016-06-01 00:00:00.000,54658,...,18.0,18.0,20.0,20.0,20.0,19.0,17.0,18.0,NaN,NaN
4,VENCIDO,ESCUELA DE INGENIERIAS,INGENIERIA DE SISTEMAS,BOGOTA CENTRO,PRESENCIAL,2011,1TC03,421,2011-02-06 00:00:00.000,2383,...,21.0,20.0,19.0,20.0,21.0,20.0,20.0,20.0,19.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA RESTREPO,PRESENCIAL,1034,PT401,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
43,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA SUBA,PRESENCIAL,1034,PT501,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
44,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,NEIVA,PRESENCIAL,1030,PT091,13435,2014-08-19 00:00:00,6124,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
45,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN SERVICIOS ADMINISTRATIVOS,MONTERIA,PRESENCIAL,6015,PT205,14426,2015-09-07 00:00:00,54274,...,17.0,20.0,20.0,20.0,15.0,17.0,20.0,17.0,NaN,NaN


In [49]:
df_final = df_final.applymap(lambda x: x.strip() if isinstance(x, str) else x)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2084\3064905405.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final = df_final.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [50]:
antes = df_final_pre.isna().sum()
despues = df_final.isna().sum()

comparacion = pd.DataFrame({
    'ANTES': antes,
    'DESPUES': despues,
    'DIFERENCIA': despues - antes
})

print(comparacion)

                            ANTES  DESPUES  DIFERENCIA
ESTADO                          0        0           0
ESCUELAS                        0        0           0
PROGRAMA_PROFESIONAL            0        0           0
SEDE                            1        1           0
MODALIDAD                       0        0           0
PENSUM                          0        0           0
COD_PROG_TECNICO                1        1           0
RESOLUCION_TECNICO              0        0           0
FECHA_INICIO_TEC              177      177           0
CODIGO_SNIES_TECNICO            1        1           0
FECHA_FIN_TEC                 178      178           0
NOMBRE_DEL_PROGRAMA_TEC         1        1           0
TITULO_OTORGADO_TEC             1        1           0
COD_PROG_TECNOLOGO            188      188           0
RESOLUCION_TNG                  0        0           0
FECHA_INICIO_TNG              737      737           0
CODIGO_SNIES_TECNOLOGICO      188      188           0
FECHA_FIN_

In [51]:
for col in df_final.select_dtypes(include=['object', 'string']).columns:
    df_final[col] = (
        df_final[col]
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.strip()
    )

In [52]:
df_final

,ESTADO,ESCUELAS,PROGRAMA_PROFESIONAL,SEDE,MODALIDAD,PENSUM,COD_PROG_TECNICO,RESOLUCION_TECNICO,FECHA_INICIO_TEC,CODIGO_SNIES_TECNICO,...,2C,3C,4C,5C,6C,7C,8C,9C,10C,11C
0,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,COMUNICACION SOCIAL,BARRANQUILLA,PRESENCIAL,3031,1TQ12,16932,2016-08-22 00:00:00.000,105792,...,16.0,19.0,17.0,19,19,16,17,17,nan,nan
1,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3024,1TC11,3567,2009-06-01 00:00:00.000,54658,...,18.0,19.0,20.0,18,19,18,20,18,20,nan
2,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3027,1TC11,3567,2009-06-01 00:00:00.000,54658,...,20.0,19.0,20.0,18,21,21,18,19,18,nan
3,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3028,1TC11,10711,2016-06-01 00:00:00.000,54658,...,18.0,18.0,20.0,20,20,19,17,18,nan,nan
4,VENCIDO,ESCUELA DE INGENIERIAS,INGENIERIA DE SISTEMAS,BOGOTA CENTRO,PRESENCIAL,2011,1TC03,421,2011-02-06 00:00:00.000,2383,...,21.0,20.0,19.0,20,21,20,20,20,19,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA RESTREPO,PRESENCIAL,1034,PT401,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20,17,14,17,16,nan,nan
43,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA SUBA,PRESENCIAL,1034,PT501,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20,17,14,17,16,nan,nan
44,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,NEIVA,PRESENCIAL,1030,PT091,13435,2014-08-19 00:00:00,6124,...,18.0,19.0,19.0,20,17,14,17,16,nan,nan
45,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN SERVICIOS ADMINISTRATIVOS,MONTERIA,PRESENCIAL,6015,PT205,14426,2015-09-07 00:00:00,54274,...,17.0,20.0,20.0,20,15,17,20,17,nan,nan


# SE TRAEN LOS TIPOS DE DATOS QUE TIENE LA TABLA DE MALLAS LAURA

In [53]:
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [54]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
                    SELECT COLUMN_NAME, DATA_TYPE
                    FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = 'CUN'
                    AND TABLE_NAME  = 'MALLASLAURA'
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_mallas_datos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_mallas_datos.shape

(46, 2)

In [55]:
df_final_pre

,ESTADO,ESCUELAS,PROGRAMA_PROFESIONAL,SEDE,MODALIDAD,PENSUM,COD_PROG_TECNICO,RESOLUCION_TECNICO,FECHA_INICIO_TEC,CODIGO_SNIES_TECNICO,...,2C,3C,4C,5C,6C,7C,8C,9C,10C,11C
0,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,COMUNICACION SOCIAL,BARRANQUILLA,PRESENCIAL,3031,1TQ12,16932,2016-08-22 00:00:00.000,105792,...,16.0,19.0,17.0,19.0,19.0,16.0,17.0,17.0,NaN,NaN
1,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3024,1TC11,3567,2009-06-01 00:00:00.000,54658,...,18.0,19.0,20.0,18.0,19.0,18.0,20.0,18.0,20.0,NaN
2,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3027,1TC11,3567,2009-06-01 00:00:00.000,54658,...,20.0,19.0,20.0,18.0,21.0,21.0,18.0,19.0,18.0,NaN
3,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3028,1TC11,10711,2016-06-01 00:00:00.000,54658,...,18.0,18.0,20.0,20.0,20.0,19.0,17.0,18.0,NaN,NaN
4,VENCIDO,ESCUELA DE INGENIERIAS,INGENIERIA DE SISTEMAS,BOGOTA CENTRO,PRESENCIAL,2011,1TC03,421,2011-02-06 00:00:00.000,2383,...,21.0,20.0,19.0,20.0,21.0,20.0,20.0,20.0,19.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA RESTREPO,PRESENCIAL,1034,PT401,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
43,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA SUBA,PRESENCIAL,1034,PT501,14439,2013-10-16 00:00:00,52606,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
44,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,NEIVA,PRESENCIAL,1030,PT091,13435,2014-08-19 00:00:00,6124,...,18.0,19.0,19.0,20.0,17.0,14.0,17.0,16.0,NaN,NaN
45,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN SERVICIOS ADMINISTRATIVOS,MONTERIA,PRESENCIAL,6015,PT205,14426,2015-09-07 00:00:00,54274,...,17.0,20.0,20.0,20.0,15.0,17.0,20.0,17.0,NaN,NaN


# se definen los tipos de datos

In [56]:
map_types = {
    'varchar': 'string',
    'nvarchar': 'string',
    'char': 'string',
    'text': 'string',
    
    'int': 'Int64',
    'bigint': 'Int64',
    'smallint': 'Int64',
    
    'decimal': 'float64',
    'numeric': 'float64',
    'float': 'float64',
    
    'date': 'datetime64[ns]',
    'datetime': 'datetime64[ns]',
    'datetime2': 'datetime64[ns]'
}

In [57]:
dict_types = {}

for _, row in df_mallas_datos.iterrows():
    col = row['COLUMN_NAME']
    sql_type = row['DATA_TYPE'].lower()
    
    if sql_type in map_types:
        dict_types[col] = map_types[sql_type]

In [58]:
for col, dtype in dict_types.items():
    if col in df_final.columns:
        try:
            if 'datetime' in dtype:
                df_final[col] = pd.to_datetime(df_final[col], errors='coerce')
            else:
                df_final[col] = df_final[col].astype(dtype)
        except Exception as e:
            print(f"Error en columna {col}: {e}")

In [59]:
df_final

,ESTADO,ESCUELAS,PROGRAMA_PROFESIONAL,SEDE,MODALIDAD,PENSUM,COD_PROG_TECNICO,RESOLUCION_TECNICO,FECHA_INICIO_TEC,CODIGO_SNIES_TECNICO,...,2C,3C,4C,5C,6C,7C,8C,9C,10C,11C
0,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,COMUNICACION SOCIAL,BARRANQUILLA,PRESENCIAL,3031,1TQ12,16932,2016-08-22,105792,...,16.0,19.0,17.0,19,19,16,17,17,nan,nan
1,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3024,1TC11,3567,2009-06-01,54658,...,18.0,19.0,20.0,18,19,18,20,18,20,nan
2,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3027,1TC11,3567,2009-06-01,54658,...,20.0,19.0,20.0,18,21,21,18,19,18,nan
3,VENCIDO,ESCUELA DE DISEÑO Y COMUNICACION,DIRECCION Y PRODUCCION MEDIOS AUDIOVISUALES,BOGOTA CENTRO,PRESENCIAL,3028,1TC11,10711,2016-06-01,54658,...,18.0,18.0,20.0,20,20,19,17,18,nan,nan
4,VENCIDO,ESCUELA DE INGENIERIAS,INGENIERIA DE SISTEMAS,BOGOTA CENTRO,PRESENCIAL,2011,1TC03,421,2011-02-06,2383,...,21.0,20.0,19.0,20,21,20,20,20,19,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA RESTREPO,PRESENCIAL,1034,PT401,14439,NaT,52606,...,18.0,19.0,19.0,20,17,14,17,16,nan,nan
43,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,BOGOTA SUBA,PRESENCIAL,1034,PT501,14439,NaT,52606,...,18.0,19.0,19.0,20,17,14,17,16,nan,nan
44,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,ADMINISTRACION DE EMPRESAS,NEIVA,PRESENCIAL,1030,PT091,13435,NaT,6124,...,18.0,19.0,19.0,20,17,14,17,16,nan,nan
45,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,TECNICA PROFESIONAL EN SERVICIOS ADMINISTRATIVOS,MONTERIA,PRESENCIAL,6015,PT205,14426,NaT,54274,...,17.0,20.0,20.0,20,15,17,20,17,nan,nan


# se elimina la informacion que habia en la tabla

In [60]:
from sqlalchemy import text

with engine.connect() as connection:
    trans = connection.begin()
    try:
        sql_query = text("""
            DELETE FROM CUN.MALLASLAURA
        """)

        connection.execute(sql_query)
        trans.commit()

    except Exception as e:
        trans.rollback()
        print(e)

    finally:
        connection.close()

# se carga la nueva info en la tabla 

In [61]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'MALLASlaura'
schema_name = 'CUN'
# Send the data to sql server
df_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2084\3907153279.py:14: UserWarning: The provided table name 'MALLASlaura' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_final.to_sql(
